In [7]:
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import StackingClassifier
from joblib import parallel_backend
import joblib
import warnings
import logging

warnings.filterwarnings('ignore')
logging.getLogger('lightgbm').setLevel(logging.ERROR)

In [2]:
df = pd.read_csv('../data/cleaned_data.csv')

## Preprocessing

In [3]:
X = df.drop(columns=['Churn'])
y = df['Churn']

In [4]:
scaler = StandardScaler()
encoder = OneHotEncoder(drop='first', sparse_output=False)

In [5]:
num_cols = ['Age','Tenure', 'Usage Frequency', 'Support Calls','Payment Delay','Total Spend','Last Interaction']
cat_cols = ['Gender','Subscription Type','Contract Length']
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ])

## Modeling

In [8]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))])


if __name__ == "__main__":
    with parallel_backend('threading'):
        scores = cross_val_score(pipeline, X, y, cv=5, n_jobs=-1)
    print(scores.mean())

0.8418683900782871


In [9]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))])


if __name__ == "__main__":
    with parallel_backend('threading'):
        scores = cross_val_score(pipeline, X, y, cv=5, n_jobs=-1)
    print(scores.mean())

0.9345790381509731


In [10]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier())])

if __name__ == "__main__":
    with parallel_backend('threading'):
        scores = cross_val_score(pipeline, X, y, cv=5, n_jobs=-1)
    print(scores.mean())

0.9329302058404692


In [11]:



pipeline_lgbm = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=20,
    random_state=42,
    n_jobs=-1,
    verbose=-1
))
])

scores = cross_val_score(pipeline_lgbm, X, y, cv=5)
print(f"LGBM : {scores.mean():.4f}")

LGBM : 0.9364


In [12]:


estimators = [
    ('lgbm', LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63, random_state=42, n_jobs=1, verbose=-1)),
    ('rf', RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42, n_jobs=1))
]

stack = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(),
    cv=5,
    n_jobs=1
)

pipeline_stack = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', stack)
])

with parallel_backend('threading'):
    scores = cross_val_score(pipeline_stack, X, y, cv=5)

print(f"Stacking: {scores.mean():.4f}")

Stacking: 0.9362


In [ ]:
pipeline_lgbm.fit(X, y)
joblib.dump(pipeline_lgbm, 'model.pkl')

['model.pkl']